In [2]:
import os
import json
import warnings
import numpy as np
import xarray as xr
import proplot as pplt
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [11]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR = CONFIGS['filepaths']['splits']
LATRANGE  = CONFIGS['domain']['latrange']
LONRANGE  = CONFIGS['domain']['lonrange']
SPLIT     = 'valid'

In [12]:
# Load z-score stats
with open(os.path.join(SPLITSDIR, 'stats.json'), 'r', encoding='utf-8') as f:
    stats = json.load(f)

# Load normalized (z-scored) split
with xr.open_dataset(os.path.join(SPLITSDIR, f'norm_{SPLIT}.h5'), engine='h5netcdf') as ds:
    # lf is static (no time dim); shf varies with time
    lf_z  = ds['lf'].load()   # (lat, lon) or (time, lat, lon)
    shf_z = ds['shf'].load()  # (time, lat, lon)
    lat   = ds['lat'].values
    lon   = ds['lon'].values

# If lf has no time dimension, broadcast to match shf
if 'time' not in lf_z.dims:
    lf_z_full = lf_z.expand_dims(time=shf_z.time).broadcast_like(shf_z)
else:
    lf_z_full = lf_z

print('lf_z dims:', lf_z.dims, '| shape:', lf_z.shape)
print('shf_z dims:', shf_z.dims, '| shape:', shf_z.shape)
print(f'lf_z  range: [{float(lf_z.min()):.2f}, {float(lf_z.max()):.2f}]')
print(f'shf_z range: [{float(shf_z.min()):.2f}, {float(shf_z.max()):.2f}]')
print(f'lf  mean={stats.get("lf_mean","N/A"):.3f}  std={stats.get("lf_std","N/A"):.3f}')
print(f'shf mean={stats.get("shf_mean","N/A"):.3f}  std={stats.get("shf_std","N/A"):.3f}')

lf_z dims: ('lat', 'lon') | shape: (21, 31)
shf_z dims: ('lat', 'lon', 'time') | shape: (21, 31, 2208)
lf_z  range: [-0.66, 1.60]
shf_z range: [-11.67, 3.24]
lf  mean=0.293  std=0.441
shf mean=-13.909  std=42.259


In [13]:
# Compute derived quantities at each (time, lat, lon)
max_term = xr.apply_ufunc(np.maximum, lf_z_full, shf_z)   # max(lf_z, shf_z)
abs_term = abs(lf_z_full - shf_z)                          # abs(lf_z - shf_z)
lf_wins  = (lf_z_full > shf_z).astype(float)              # 1 where lf dominates

# Time-average each quantity
lf_z_mean   = lf_z if 'time' not in lf_z.dims else lf_z.mean('time')
shf_z_mean  = shf_z.mean('time')
max_mean     = max_term.mean('time')
abs_mean     = abs_term.mean('time')
lf_wins_frac = lf_wins.mean('time')  # fraction of time steps where lf_z > shf_z

# Native-unit time means for reference
lf_mean_val  = stats.get('lf_mean',  0.0)
lf_std_val   = stats.get('lf_std',   1.0)
shf_mean_val = stats.get('shf_mean', 0.0)
shf_std_val  = stats.get('shf_std',  1.0)

lf_native  = lf_z_mean  * lf_std_val  + lf_mean_val
shf_native = shf_z_mean * shf_std_val + shf_mean_val

print('Time-mean lf_z:', float(lf_z_mean.mean()), '(should be ~0 if static with zero mean after z-score)')
print('Time-mean shf_z:', float(shf_z_mean.mean()))

Time-mean lf_z: -7.031700022253062e-08 (should be ~0 if static with zero mean after z-score)
Time-mean shf_z: 0.013705512508749962


In [14]:
# ── Panel 1: Raw inputs in native units ────────────────────────────────────────
# Shows what lf and shf actually look like before z-scoring
fig, axs = pplt.subplots(nrows=1, ncols=2, proj='cyl')
axs.format(coast=True, borders=False, latlim=LATRANGE, lonlim=LONRANGE,
           latlines=5, lonlines=5, grid=False)

m1 = axs[0].pcolormesh(lon, lat, lf_native, cmap='yellows4', vmin=0, vmax=1, levels=11)
axs[0].format(title='LF (land fraction)')
fig.colorbar(m1, loc='b', label='LF (fraction)', ticks=0.2)

shf_lim = max(abs(float(shf_native.min())), abs(float(shf_native.max())))
m2 = axs[1].pcolormesh(lon, lat, shf_native, cmap='ColdHot', vmin=-shf_lim, vmax=shf_lim, levels=21)
axs[1].format(title='SHF — time mean (W m$^{-2}$)', abc='(b)')
fig.colorbar(m2, loc='b', label='SHF (W m$^{-2}$)')

fig.format(suptitle='Native-unit inputs')
pplt.show()
# fig.save('../figs/surface_terms_native.jpg')

KeyError: None

Figure(refwidth=2.5)

In [16]:
# ── Panel 2: Z-scored inputs ───────────────────────────────────────────────────
# Shows the space in which the SR equation actually operates
# vmax_z = max(
#     abs(float(lf_z_mean.min())), abs(float(lf_z_mean.max())),
#     abs(float(shf_z_mean.min())), abs(float(shf_z_mean.max())))

# fig, axs = pplt.subplots(nrows=1, ncols=2, proj='cyl', figwidth=6.5, share=False, wspace=0)
# axs.format(coast=True, borders=False, latlim=LATRANGE, lonlim=LONRANGE,
#            latlines=5, lonlines=5, grid=False)

# m1 = axs[0].pcolormesh(lon, lat, lf_z_mean, cmap='ColdHot', vmin=-vmax_z, vmax=vmax_z, levels=21)
# axs[0].format(title='LF (z-scored)', abc='(a)')
# fig.colorbar(m1, loc='b', label='Standard deviations')

# m2 = axs[1].pcolormesh(lon, lat, shf_z_mean, cmap='ColdHot', vmin=-vmax_z, vmax=vmax_z, levels=21)
# axs[1].format(title='SHF — time mean (z-scored)', abc='(b)')
# fig.colorbar(m2, loc='b', label='Standard deviations')

# fig.format(suptitle='Z-scored inputs (model input space)')
# pplt.show()
# fig.save('../figs/surface_terms_zscored.jpg')

KeyError: None

Figure(figwidth=6.5)

In [ ]:
# ── Panel 3: max(lf,shf) and abs(lf-shf), time-mean ──────────────────────────
# The two candidate SR-HI surface terms in z-scored space
fig, axs = pplt.subplots(nrows=1, ncols=2, proj='cyl', figwidth=6.5, share=False, wspace=0)
axs.format(coast=True, borders=False, latlim=LATRANGE, lonlim=LONRANGE,
           latlines=5, lonlines=5, grid=False)

vmax_max = float(max_mean.quantile(0.99))
m1 = axs[0].pcolormesh(lon, lat, max_mean, cmap='Grays', vmin=0, vmax=vmax_max, levels=11)
axs[0].format(title='max(LF, SHF) — time mean', abc='(a)')
fig.colorbar(m1, loc='b', label='z-score units')

vmax_abs = float(abs_mean.quantile(0.99))
m2 = axs[1].pcolormesh(lon, lat, abs_mean, cmap='Grays', vmin=0, vmax=vmax_abs, levels=11)
axs[1].format(title='|LF − SHF| — time mean', abc='(b)')
fig.colorbar(m2, loc='b', label='z-score units')

fig.format(suptitle='SR-HI surface term candidates (z-scored)')
pplt.show()
fig.save('../figs/surface_terms_derived.jpg')

In [ ]:
# ── Panel 4: Dominance map — fraction of time steps where lf_z > shf_z ────────
# Answers: "where does land fraction dominate in max(lf, shf)?"
fig, axs = pplt.subplots(nrows=1, ncols=1, proj='cyl', figwidth=4.0)
axs.format(coast=True, borders=False, latlim=LATRANGE, lonlim=LONRANGE,
           latlines=5, lonlines=5, grid=False,
           title='Fraction of time: LF wins max(LF, SHF)')

m = axs.pcolormesh(lon, lat, lf_wins_frac,
                   cmap='DryWet', vmin=0, vmax=1, levels=11)
fig.colorbar(m, loc='b', label='Fraction of time steps', ticks=0.2)

pplt.show()
fig.save('../figs/surface_terms_dominance.jpg')

In [ ]:
# ── Panel 5: Full 2×2 summary ──────────────────────────────────────────────────
# Lay out all four derived quantities for easy comparison
vmax_z = max(
    abs(float(lf_z_mean.min())), abs(float(lf_z_mean.max())),
    abs(float(shf_z_mean.min())), abs(float(shf_z_mean.max())))

fig, axs = pplt.subplots(nrows=2, ncols=2, proj='cyl', figwidth=6.5,
                         share=False, wspace=0, hspace=0.5)
axs.format(coast=True, borders=False, latlim=LATRANGE, lonlim=LONRANGE,
           latlines=5, lonlines=5, grid=False)

# (a) z-scored lf
m1 = axs[0].pcolormesh(lon, lat, lf_z_mean, cmap='ColdHot',
                        vmin=-vmax_z, vmax=vmax_z, levels=21)
axs[0].format(title='LF (z-scored)', abc='(a)')
fig.colorbar(m1, ax=axs[0], loc='r', label='σ')

# (b) z-scored shf (time mean)
m2 = axs[1].pcolormesh(lon, lat, shf_z_mean, cmap='ColdHot',
                        vmin=-vmax_z, vmax=vmax_z, levels=21)
axs[1].format(title='SHF mean (z-scored)', abc='(b)')
fig.colorbar(m2, ax=axs[1], loc='r', label='σ')

# (c) max(lf,shf) time mean
vmax_max = float(max_mean.quantile(0.99))
m3 = axs[2].pcolormesh(lon, lat, max_mean, cmap='Grays',
                        vmin=0, vmax=vmax_max, levels=11)
axs[2].format(title='max(LF, SHF) — mean', abc='(c)')
fig.colorbar(m3, ax=axs[2], loc='r', label='σ')

# (d) fraction lf wins
m4 = axs[3].pcolormesh(lon, lat, lf_wins_frac, cmap='DryWet',
                        vmin=0, vmax=1, levels=11)
axs[3].format(title='Fraction: LF wins', abc='(d)')
fig.colorbar(m4, ax=axs[3], loc='r', label='Fraction', ticks=0.2)

pplt.show()
fig.save('../figs/surface_terms_summary.jpg')

In [ ]:
# ── Diagnostics: print crossover statistics ────────────────────────────────────
# Understand how much the two variables overlap in value
lf_arr  = lf_z_full.values.ravel()
shf_arr = shf_z.values.ravel()
fin     = np.isfinite(lf_arr) & np.isfinite(shf_arr)

lf_wins_pct = 100 * (lf_arr[fin] > shf_arr[fin]).mean()
print(f'lf_z > shf_z (i.e. LF wins) at {lf_wins_pct:.1f}% of all (time, lat, lon) samples')
print()
print(f'lf_z  : mean={lf_arr[fin].mean():.3f}  std={lf_arr[fin].std():.3f}  '
      f'min={lf_arr[fin].min():.2f}  max={lf_arr[fin].max():.2f}')
print(f'shf_z : mean={shf_arr[fin].mean():.3f}  std={shf_arr[fin].std():.3f}  '
      f'min={shf_arr[fin].min():.2f}  max={shf_arr[fin].max():.2f}')
print()

# Crossover: at what native lf does lf_z ≈ mean(shf_z)?
shf_z_global_mean = float(shf_arr[fin].mean())
crossover_lf_z    = shf_z_global_mean
crossover_lf_native = crossover_lf_z * lf_std_val + lf_mean_val
print(f'lf_z = mean(shf_z) = {shf_z_global_mean:.3f}')
print(f'  → crossover in native lf units ≈ {crossover_lf_native:.3f}')